The code in this file creates a set of xyz files with model polynuclear ThxOy clusters from a bigger single cluster which is stored in 'Th40.xyz' file. In this paper, Th40 cluster is constructed from the Ce40 crystal structure (CSD refcode KENGUI) by extracting a single molecular cluster from the crytsl structure, stripping all the organic fragments and substituting Ce atoms with Th atoms.

The decision on how to slice an initial cluster is made based on the structure catalogue file {structure_catalogue.txt} created by structure_catalogue_maker() function from this paper {https://doi.org/10.1038/s41524-022-00896-3}. 

As the atoms to be dropped from the initial big structure when constructing each smaller structure are selected randomly, some of the resulting structures may not be connected and consist of several atom 'islands'. In that case, the island containing the largest number of heavy atoms is retained with all coordinated oxygen atoms, while the smaller islands are dropped.

The names of the prepared files are formatted so that they contain a number of Th atoms in the cluster as the first symbol for easier labelling of data before the network training.

In [ ]:
import numpy as np
import os

In [ ]:
def make_xyz_catalogue(catalogue, initial_xyz, treshold_ThO, treshold_ThTh):
    
    with open(catalogue, 'r') as f:
        strings = f.readlines()
    arrays = []
    for s in strings:
        # Split the string into a list of numbers
        numbers = list(map(float, s.split()))
        arrays.append(np.array(numbers))

    with open(initial_xyz, 'r') as f:
        lines = f.readlines()

    # Get the number of atoms from the xyz file
    num_atoms = int(lines[0])

    # Iterate over the arrays
    for array, k in zip(arrays, range(len(arrays)+1)):
        # Create a list to store the modified xyz lines
        new_lines = [f"{num_atoms}\n", lines[1]]

        # Iterate over the atoms in the xyz file
        for i, line in enumerate(lines[2:]):
            # If the index is less than the length of the array, process the atom according to the array value
            if i < len(array) and array[i] == 1 or i >= len(array):
                new_lines.append(line)

        Th_coordinates = []
        for line in new_lines[2:]:
            fields = line.split()
            if fields and fields[0] == "Th":
                Th_coordinates.append([float(fields[1]), float(fields[2]), float(fields[3])])

        Th_groups = []
        while Th_coordinates:
            Th_group = [Th_coordinates.pop()]
            for Th_coordinate in Th_coordinates:
                distance = np.linalg.norm(np.array(Th_group[0]) - np.array(Th_coordinate))
                if distance < treshold_ThTh:
                    Th_group.append(Th_coordinate)
            Th_groups.append(Th_group)

        final_result = []
        for element in Th_groups:
            found = False
            for result in final_result:
                if set(map(tuple, element)) & set(map(tuple, result)):
                    result.extend([e for e in element if e not in result])
                    found = True
                    break
            if not found:
                final_result.append(element)

        if Th_groups:

            largest_Th_group = max(final_result, key=lambda x: len(x))
            
            with open(f'{initial_xyz}_{k}.xyz', 'w') as new_f:
                new_f.write(new_lines[0])
                new_f.write(new_lines[1])
                for Th_coordinate in largest_Th_group:
                    new_f.write("Th " + " ".join([str(x) for x in Th_coordinate]) + "\n")
                for line in lines[2:]:
                    fields = line.split()
                    if fields and fields[0] == "O":
                        O_coordinate = [float(fields[1]), float(fields[2]), float(fields[3])]
                        for Th_coordinate in largest_Th_group:
                            distance = np.linalg.norm(np.array(Th_coordinate) - np.array(O_coordinate))
                            if distance < treshold_ThO:
                                new_f.write(line)
                                break


In [ ]:
os.chdir('/your/project/directory/')

In [ ]:
make_xyz_catalogue(catalogue='structure_catalogue.txt', initial_xyz='th40.xyz', 
                   treshold_ThO=3.0, treshold_ThTh=5.2)

In [ ]:
directory = "/your/project/directory/th_groups/"

for filename, i in zip(os.listdir(directory), range(10001)):
    path = os.path.join(directory, filename)
    if path.startswith('/your/project/directory/th_groups/th40'):
        with open(path, 'r', encoding='ISO-8859-1') as f:
            lines = f.readlines()
            Th_count = sum(1 for line in lines if line.startswith("Th"))
            new_filename = f"{Th_count}_{i}.xyz"
            new_path = os.path.join(directory, new_filename)
            os.rename(path, new_path)